# EV Charging Station Analysis
**Dataset:** `detailed_ev_charging_stations.csv`  
A complete pipeline: data loading → EDA → machine learning → export.

## Block 1 — Setup & Imports

In [ ]:
# ── Standard data libraries ──────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine learning ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Global plot style ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('All libraries imported successfully ✓')

## Block 2 — Data Loading & Inspection

In [ ]:
# ── Load the dataset ──────────────────────────────────────────────────────────
# Update the path below if the file lives elsewhere
CSV_PATH = r'C:\Users\acer\Downloads\archive (1)\detailed_ev_charging_stations.csv'

df = pd.read_csv(CSV_PATH)

# ── Shape ─────────────────────────────────────────────────────────────────────
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n')

# ── Column types & non-null counts ────────────────────────────────────────────
print('--- DataFrame Info ---')
df.info()

# ── First 3 rows ──────────────────────────────────────────────────────────────
print('\n--- First 3 Rows ---')
display(df.head(3))

# ── Missing values ────────────────────────────────────────────────────────────
print('\n--- Missing Values per Column ---')
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('No missing values found — dataset is clean ✓')
else:
    display(missing.to_frame(name='Missing Count'))

## Block 3 — Exploratory Data Analysis & Visualizations

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Chart 1 — Usage Stats vs. Distance to City  (Scatter plot)
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 5))

scatter = ax.scatter(
    df['Distance to City (km)'],
    df['Usage Stats (avg users/day)'],
    c=df['Charging Capacity (kW)'],
    cmap='viridis',
    alpha=0.6,
    edgecolors='none',
    s=40,
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Charging Capacity (kW)', fontsize=10)

ax.set_title('Usage Stats vs. Distance to City', fontsize=14, fontweight='bold')
ax.set_xlabel('Distance to City (km)', fontsize=11)
ax.set_ylabel('Avg Users / Day', fontsize=11)
plt.tight_layout()
plt.savefig('chart1_usage_vs_distance.png', bbox_inches='tight')
plt.show()
print('Chart 1 saved ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Chart 2 — Average Cost (USD/kWh) across Charger Types  (Bar chart)
# ═══════════════════════════════════════════════════════════════════════════════
avg_cost = (
    df.groupby('Charger Type')['Cost (USD/kWh)']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7, 4))
bars = sns.barplot(
    data=avg_cost,
    x='Charger Type',
    y='Cost (USD/kWh)',
    palette='Blues_d',
    ax=ax,
)

# Annotate bar values
for bar in bars.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f'${bar.get_height():.3f}',
        ha='center', va='bottom', fontsize=10,
    )

ax.set_title('Average Cost (USD/kWh) by Charger Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Charger Type', fontsize=11)
ax.set_ylabel('Avg Cost (USD/kWh)', fontsize=11)
plt.tight_layout()
plt.savefig('chart2_avg_cost_by_charger_type.png', bbox_inches='tight')
plt.show()
print('Chart 2 saved ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Chart 3 — Top Station Operators Market Share  (Horizontal bar chart)
# ═══════════════════════════════════════════════════════════════════════════════
top_operators = (
    df['Station Operator']
    .value_counts()
    .head(10)
    .reset_index()
)
top_operators.columns = ['Station Operator', 'Count']

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    data=top_operators,
    y='Station Operator',
    x='Count',
    palette='rocket_r',
    ax=ax,
)

for bar in ax.patches:
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f'{int(bar.get_width())}',
        va='center', fontsize=10,
    )

ax.set_title('Top 10 Station Operators by Number of Stations', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Stations', fontsize=11)
ax.set_ylabel('Station Operator', fontsize=11)
plt.tight_layout()
plt.savefig('chart3_top_operators.png', bbox_inches='tight')
plt.show()
print('Chart 3 saved ✓')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Chart 4 — Renewable Energy Source Breakdown  (Count plot)
# ═══════════════════════════════════════════════════════════════════════════════
renewable_counts = df['Renewable Energy Source'].value_counts().reset_index()
renewable_counts.columns = ['Renewable Energy Source', 'Count']

fig, ax = plt.subplots(figsize=(6, 4))
palette = {'Yes': '#2ecc71', 'No': '#e74c3c'}

sns.countplot(
    data=df,
    x='Renewable Energy Source',
    palette=palette,
    order=['Yes', 'No'],
    ax=ax,
)

total = len(df)
for bar in ax.patches:
    count = int(bar.get_height())
    pct = count / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.005,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=11,
    )

ax.set_title('Renewable Energy Source Breakdown', fontsize=14, fontweight='bold')
ax.set_xlabel('Uses Renewable Energy?', fontsize=11)
ax.set_ylabel('Number of Stations', fontsize=11)
plt.tight_layout()
plt.savefig('chart4_renewable_energy.png', bbox_inches='tight')
plt.show()
print('Chart 4 saved ✓')

## Block 4 — Predictive Machine Learning Model

In [ ]:
# ── Step 1: Create binary target ──────────────────────────────────────────────
median_usage = df['Usage Stats (avg users/day)'].median()
df['High_Utilization'] = (df['Usage Stats (avg users/day)'] > median_usage).astype(int)

print(f'Median usage: {median_usage:.1f} users/day')
print(f'High Utilization (1): {df["High_Utilization"].sum():,} stations')
print(f'Low  Utilization (0): {(df["High_Utilization"] == 0).sum():,} stations')

# ── Step 2: Select numerical features ────────────────────────────────────────
FEATURES = [
    'Charging Capacity (kW)',
    'Cost (USD/kWh)',
    'Distance to City (km)',
    'Parking Spots',
    'Reviews (Rating)',
]
TARGET = 'High_Utilization'

X = df[FEATURES]
y = df[TARGET]

# ── Step 3: Train / test split (80 / 20) ─────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Step 4: Feature scaling ───────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Step 5: Train Logistic Regression ────────────────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

# ── Step 6: Predictions & metrics ────────────────────────────────────────────
y_pred = model.predict(X_test_sc)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)

print('\n═══ Model Performance ═══')
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')

# ── Step 7: Confusion matrix plot ────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low (0)', 'High (1)'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Logistic Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chart5_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Confusion matrix saved ✓')

## Block 5 — Priority Scoring Export

In [ ]:
# ── Generate utilization probabilities for the FULL dataset ──────────────────
X_all_sc = scaler.transform(df[FEATURES])
df['Utilization_Probability'] = model.predict_proba(X_all_sc)[:, 1]

# ── Classify into priority tiers ─────────────────────────────────────────────
def assign_priority(prob):
    if prob >= 0.75:
        return 'HIGH'
    elif prob >= 0.50:
        return 'MEDIUM'
    else:
        return 'LOW'

df['Priority_Tier'] = df['Utilization_Probability'].apply(assign_priority)

# ── Build export table ────────────────────────────────────────────────────────
export_cols = [
    'Station ID',
    'Address',
    'Charger Type',
    'Station Operator',
    'Charging Capacity (kW)',
    'Cost (USD/kWh)',
    'Distance to City (km)',
    'Parking Spots',
    'Reviews (Rating)',
    'Usage Stats (avg users/day)',
    'High_Utilization',
    'Utilization_Probability',
    'Priority_Tier',
]

priority_df = (
    df[export_cols]
    .sort_values('Utilization_Probability', ascending=False)
    .reset_index(drop=True)
)
priority_df['Utilization_Probability'] = priority_df['Utilization_Probability'].round(4)

# ── Export to CSV ─────────────────────────────────────────────────────────────
OUTPUT_PATH = 'ev_station_priority_scores.csv'
priority_df.to_csv(OUTPUT_PATH, index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'Exported {len(priority_df):,} rows → "{OUTPUT_PATH}" ✓\n')
print('Priority Tier Distribution:')
print(priority_df['Priority_Tier'].value_counts().to_string())
print()
display(priority_df.head(10))